In [ ]:
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
analyze_path = os.path.join(parent_dir, "utils")

os.chdir(analyze_path)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from utils import get_grid, read_data, read_taiwan_specific

combined_data = read_data()
taiwan, grid_filter = read_taiwan_specific()

from config import col_translation

In [ ]:
labels_df = pd.read_csv("../ComputedDataV7/Grid/result_df.csv")

final_full = pd.read_csv("../ComputedDataV7/ForModel/final_data_full.csv")
final_grid = pd.read_csv("../ComputedDataV7/Grid/trend_grid.csv")
final_grid = final_grid[final_grid['num_accidents'] > 0]
final_grid.reset_index(inplace=True, drop=True)

final_full = pd.concat([final_full, final_grid[['target_label']]], axis=1)
final_full.drop(columns=['hotspot', 'COUNTYNAME', 'TOWNNAME'], inplace=True, errors='ignore')
final_full.rename(columns=col_translation, inplace=True)

final_full = final_full.merge(labels_df, on='grid_id', how='inner')

from sklearn.preprocessing import MinMaxScaler
rescale_cols = [
    'num_mrt', 'lag_num_mrt', 'num_parking', 'lag_num_parking',
    'num_youbike', 'lag_num_youbike', 'num_speed_diff', 'lag_num_speed_diff',
    'num_bus_stop', 'lag_num_bus_stop'
]
scaler = MinMaxScaler()
final_full[rescale_cols] = scaler.fit_transform(final_full[rescale_cols])

df_cluster = {
    0: final_full[final_full['DTW_Cluster'] == 0].copy(),
    1: final_full[final_full['DTW_Cluster'] == 1].copy(),
    2: final_full[final_full['DTW_Cluster'] == 2].copy()
}

In [ ]:
import shap

os.makedirs("../ComputedDataV7/Shap_Top3_Interactions", exist_ok=True)
os.makedirs("../ComputedDataV7/Saved_Models", exist_ok=True)

drop_cols = [
    'target_label', 'hotspot', 'accident_indices', 'geometry', 'grid_id',
    'target_label', 'DTW_Cluster', 'is_peak_hotspot', 'is_off_peak_hotspot',
    'label_OffPeak2Peak', 'label_Peak2OffPeak',

    '路面狀況-路面鋪裝名稱', '路面狀況-路面缺陷名稱',
    '道路障礙-障礙物名稱', '道路障礙-視距品質名稱', '道路障礙-視距名稱'
]

model_configs = [
    # Cluster 0
    {"name": "M0_OP2P_Emerg", "cluster": 0, "label_col": "label_OffPeak2Peak", "targets": ['Stable_Safe', 'Emergent'], "focus": "Emergent"},
    {"name": "M0_OP2P_Diss",  "cluster": 0, "label_col": "label_OffPeak2Peak", "targets": ['Persistent', 'Dissipated'], "focus": "Dissipated"},
    {"name": "M0_P2OP_Emerg", "cluster": 0, "label_col": "label_Peak2OffPeak", "targets": ['Stable_Safe', 'Emergent'], "focus": "Emergent"},
    {"name": "M0_P2OP_Diss",  "cluster": 0, "label_col": "label_Peak2OffPeak", "targets": ['Persistent', 'Dissipated'], "focus": "Dissipated"},
    
    # Cluster 1
    {"name": "M1_OP2P_Emerg", "cluster": 1, "label_col": "label_OffPeak2Peak", "targets": ['Stable_Safe', 'Emergent'], "focus": "Emergent"},
    {"name": "M1_OP2P_Diss",  "cluster": 1, "label_col": "label_OffPeak2Peak", "targets": ['Persistent', 'Dissipated'], "focus": "Dissipated"},
    {"name": "M1_P2OP_Emerg", "cluster": 1, "label_col": "label_Peak2OffPeak", "targets": ['Stable_Safe', 'Emergent'], "focus": "Emergent"},
    {"name": "M1_P2OP_Diss",  "cluster": 1, "label_col": "label_Peak2OffPeak", "targets": ['Persistent', 'Dissipated'], "focus": "Dissipated"},

    # Cluster 2
    {"name": "M2_OP2P_Emerg", "cluster": 2, "label_col": "label_OffPeak2Peak", "targets": ['Stable_Safe', 'Emergent'], "focus": "Emergent"},
    {"name": "M2_OP2P_Diss",  "cluster": 2, "label_col": "label_OffPeak2Peak", "targets": ['Persistent', 'Dissipated'], "focus": "Dissipated"},
    {"name": "M2_P2OP_Emerg", "cluster": 2, "label_col": "label_Peak2OffPeak", "targets": ['Stable_Safe', 'Emergent'], "focus": "Emergent"},
    {"name": "M2_P2OP_Diss",  "cluster": 2, "label_col": "label_Peak2OffPeak", "targets": ['Persistent', 'Dissipated'], "focus": "Dissipated"}
]

In [ ]:
import joblib
from imblearn.combine import SMOTEENN
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

def train_model(X, y, target_types, model_type='lgbm', random_state=42):

    mask = y.isin(target_types)
    X_type = X[mask]
    y_type = y[mask]

    X_train_type, X_test_type, y_train_type, y_test_type = train_test_split(
        X_type, y_type, test_size=0.2, random_state=random_state, stratify=y_type
    )

    smote_enn = SMOTEENN(random_state=random_state)
    X_train_resampled, y_train_resampled = smote_enn.fit_resample(X_train_type, y_train_type)

    if model_type == 'rf':
        base_model = RandomForestClassifier(random_state=random_state)
        # param_grid = {
        #     'n_estimators': [100, 300],
        #     'max_depth': [10, None],
        #     'min_samples_split': [2, 10],
        #     'criterion': ['gini', 'entropy']
        # }
        param_grid = {
            'n_estimators': [300],
            'max_depth': [None],
            'min_samples_split': [2],
            'criterion': ['gini']
        }
    elif model_type == 'lgbm':
        base_model = LGBMClassifier(random_state=random_state, verbose=-1)
        # param_grid = {
        #     'n_estimators': [100, 300],
        #     'max_depth': [10, 20], 
        #     'learning_rate': [0.01, 0.1],
        #     'num_leaves': [30, 100]
        # }
        param_grid = {
            'n_estimators': [300],
            'max_depth': [10], 
            'learning_rate': [0.01],
            'num_leaves': [30]
        }
    else:
        raise ValueError("need rf or lgbm")

    print(f"{model_type.upper()}...")
    grid_search = GridSearchCV(
        base_model,
        param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )

    grid_search.fit(X_train_resampled, y_train_resampled)

    print(f"最佳參數: {grid_search.best_params_}")
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_type)

    print(classification_report(y_test_type, y_pred))

    return best_model, X_test_type, y_test_type

In [ ]:
for config in model_configs:
    model_name = config["name"]
    c_id = config["cluster"]
    label_col = config["label_col"]
    target_types = config["targets"]
    focus_label = config["focus"]
    
    print(f"\n{'='*50}")
    print(f"開始訓練模型: {model_name} | Cluster: {c_id} | 目標: {focus_label}")

    df_current = df_cluster[c_id].dropna(subset=[label_col])
    X_current = df_current.drop(columns=[col for col in drop_cols if col in df_current.columns])
    print(X_current.shape)
    y_current = df_current[label_col]
    print(y_current.value_counts())

    try:
        best_rf, X_test, y_test = train_model(X_current, y_current, target_types, model_type='lgbm')
    except ValueError as e:
        print(f"{model_name} 訓練失敗: {e}")
        continue

    joblib.dump(best_rf, f"../ComputedDataV7/Saved_Models/{model_name}.joblib")

    # --- SHAP 交互作用分析 ---
    print(f"正在計算 {model_name} 的 SHAP 交互作用...")
    X_sample = X_test.sample(min(1000, len(X_test)), random_state=42)
    explainer = shap.TreeExplainer(best_rf)
    interaction_values = explainer.shap_interaction_values(X_sample)
    
    # 處理正負號與目標類別對齊
    class_idx = list(best_rf.classes_).index(focus_label)
    if isinstance(interaction_values, list):
        target_interaction = interaction_values[class_idx]
    else:
        target_interaction = interaction_values if class_idx == 1 else -1 * interaction_values
        
    bike_idx = list(X_test.columns).index("lag_num_youbike")
    interaction_strengths = np.abs(target_interaction[:, bike_idx, :]).mean(0)
    interaction_strengths[bike_idx] = 0 # 排除自己跟自己的交互
    
    top_3_indices = np.argsort(interaction_strengths)[-3:][::-1]
    top_3_features = X_test.columns[top_3_indices]

    print(f"{model_name} 中與 YouBike 交互最強的 Top 3: {list(top_3_features)}")

    # 畫圖並存檔
    for feature in top_3_features:
        plt.figure(figsize=(8, 6))
        shap.dependence_plot(
            ("lag_num_youbike", feature),
            target_interaction,
            X_sample,
            show=False
        )
        plt.title(f"{model_name} | Focus: {focus_label}\nBike Interaction with {feature}")
        
        safe_feature_name = feature.replace("/", "_").replace(" ", "_")
        plt.savefig(f"../ComputedDataV7/Shap_Top3_Interactions/{model_name}_Bike_x_{safe_feature_name}.png", bbox_inches='tight')
        plt.close()